### SETUP

In [1]:
from pyspark.sql import SparkSession
from delta import *

In [2]:
builder = SparkSession.builder.appName("Local_Lakehouse") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")\
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "2g") \
    .config("spark.sql.caseSensitive", "true")

In [3]:
spark = configure_spark_with_delta_pip(builder).getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/10 17:24:15 WARN Utils: Your hostname, LAPTOP-MSL0MUJB, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/10 17:24:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/harsh/.ivy2.5.2/cache
The jars for the packages stored in: /home/harsh/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-923d1a40-e3c6-4db0-bb0c-b2532b2c747f;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.1.0 in central
	found io.delta#delta-storage;4.1.0 in central
	found io.unitycatalog#unitycatalog-client;0.4.0 in central
	found org.slf4j#slf4j-api;2.0.13 in 

### Reading data

In [4]:
df = spark.read \
    .format("json") \
    .load("/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/initial_firehose/")

In [5]:
spark.conf.set('spark.sql.repl.eagerEval.enabled', True)

In [6]:
df.show(truncate=False,n=10)

+---------------+-------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------+----------------------------

### Filtering english posts

In [7]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [8]:
df=df.filter(array_contains(col("commit.record.langs"),"en"))

In [9]:
df.show(n=20,truncate=False)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Filtering posts

In [10]:
post_df=df.filter(col("commit.collection")=="app.bsky.feed.post")
post_df.show(truncate=False,n=10)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [11]:
post_df=post_df.filter(col("commit.operation")=="create")
post_df.show(truncate=False,n=10)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Finding root_id

In [12]:
def find_root_id(root_uri):
    if root_uri is None:
        return "none_error"
    root_id=""
    i=5
    n=len(root_uri)
    if n<=5:
        return "error"
    while root_uri[i]!='/':
        root_id+=root_uri[i]
        i+=1
        if i>=n:
            return "error"
    return root_id
find_root_id_udf = udf(find_root_id, StringType())

In [13]:
post_df=post_df.withColumn("root_id",find_root_id_udf(col("commit.record.reply.root.uri")))
post_df.show(truncate=False,n=100)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
    #i have list of posts whose commit.record.reply.root contains
'''
American Megachurch Pastors need to be stopped. I just watched a clip from a CBS thing from a decade ago and they had this one pastor who, in succession had living props of:\n\nHis wife\n\nA sheep\n\nA camel\n\nand\n\nA fishing rod
'''

'\nAmerican Megachurch Pastors need to be stopped. I just watched a clip from a CBS thing from a decade ago and they had this one pastor who, in succession had living props of:\n\nHis wife\n\nA sheep\n\nA camel\n\nand\n\nA fishing rod\n'

In [15]:
# error_df=post_df.filter(col("root_id")=="error")
# error_df.show(truncate=False,n=20)

##### Conclusion: those with no root id are the root posts("none_error" in root_id column)

### Adding current_post_uri

In [16]:
post_df=post_df.withColumn("current_post_uri",concat(lit("at://"),col("did"),lit("/"),col("commit.collection"),lit("/"),col("commit.rkey")))

In [17]:
post_df.show(n=20,truncate=False)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### Making root_only df and non_root_only df

In [18]:
root_only_df=post_df.filter(col("root_id")=="none_error")

In [19]:
root_only_df.show(n=10,truncate=False)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Traceback (most recent call last):                                              
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


In [20]:
non_root_only_df=post_df.filter(col("root_id")!="none_error")

In [21]:
non_root_only_df.show(n=10,truncate=False)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Traceback (most recent call last):                                              
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


### People who commented on their own post

In [22]:
non_root_only_df.filter(col("did")==col("root_id")).show(truncate=False,n=20)

+---------------+-------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Traceback (most recent call last):                                              
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 233, in manager
    code = worker(sock, authenticated)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/python/lib/pyspark.zip/pyspark/daemon.py", line 87, in worker
    outfile.flush()
BrokenPipeError: [Errno 32] Broken pipe


### Checking if non_root_only_df contains any root post

In [23]:
# non_root_only_df.filter(col("current_post_uri")==col("commit.record.reply.root.uri")).show(truncate=False,n=20)

### Start of the end

In [24]:
import pandas as pd
import requests
import time
import os
import json

In [25]:
# ---------------------------------------------------------
# 1. Setup Persistent Storage Paths & Constants
# ---------------------------------------------------------
TRACKING_FILE = "/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/processed_root_uris.txt"
RESULTS_FILE = "/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/posts_results.jsonl" 
BSKY_API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getPosts"

BATCH_SIZE = 25
# 0.15 seconds yields ~2000 requests per 5 mins (safely under the 3000 limit)
SLEEP_TIME = 0.05

processed_root_uris = set()

# Load existing URIs if the script is restarting after a crash
if os.path.exists(TRACKING_FILE):
    with open(TRACKING_FILE, "r") as f:
        processed_root_uris = set(line.strip() for line in f if line.strip())
    print(f"Loaded {len(processed_root_uris)} previously processed URIs from disk.")

# ---------------------------------------------------------
# 2. Batch Fetching Helper Function (with 429 Retry Logic)
# ---------------------------------------------------------
def fetch_posts_batch(uris_batch, retries=3):
    """Fetches a batch of posts with built-in retry logic for rate limits."""
    for attempt in range(retries):
        try:
            params = {'uris': uris_batch}
            response = requests.get(BSKY_API_URL, params=params)
            
            # Handle rate limiting explicitly
            if response.status_code == 429:
                # Use the API's requested wait time, default to 5 seconds if missing
                wait = int(response.headers.get("Retry-After", 5))
                print(f"Rate limited (429). Sleeping {wait} seconds...")
                time.sleep(wait)
                continue # Try the same batch again

            # For all other HTTP errors
            response.raise_for_status()
            return response.json().get('posts', [])

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2) # Brief pause before retrying a general network failure

    # If it fails after all retries
    return None

# ---------------------------------------------------------
# 3. Main Processing Logic
# ---------------------------------------------------------
print("Extracting unique URIs using PySpark...")

# Step A: Collect all unique URIs using Spark's distributed power
# This safely navigates the nested JSON/Struct schema, finds unique values, and drops nulls/errors
try:
    unique_uris_df = non_root_only_df.select("commit.record.reply.root.uri").distinct().dropna()
    
    # .collect() pulls the unique list from the cluster back to your main driver node
    all_root_uris = [row.uri for row in unique_uris_df.collect()]
    
except Exception as e:
    print(f"Error extracting URIs with Spark. Ensure schema matches: {e}")
    all_root_uris = []

# Filter out the URIs we've already processed in a previous crash/run
pending_uris = [uri for uri in all_root_uris if uri not in processed_root_uris]

print(f"Found {len(pending_uris)} new unique URIs to fetch.")

# ... Continue with Step B (the batch fetching loop) exactly as it was ...

# Step B: Process in chunks of 25 with safe disk-writing
with open(TRACKING_FILE, "a") as tracker_file, open(RESULTS_FILE, "a") as results_file:
    
    for i in range(0, len(pending_uris), BATCH_SIZE):
        batch = pending_uris[i:i + BATCH_SIZE]
        print(f"Fetching batch of {len(batch)} URIs ({i} to {i + len(batch)})...")
        
        posts_data = fetch_posts_batch(batch)
        
        # ONLY save and mark as processed if the fetch was completely successful
        if posts_data is not None:
            
            # 1. Save the actual JSON post data to disk immediately
            for post in posts_data:
                results_file.write(json.dumps(post) + "\n")
            results_file.flush() # FORCE write to disk
            
            # 2. Update tracking set and save URIs to disk immediately
            for uri in batch:
                processed_root_uris.add(uri)
                tracker_file.write(f"{uri}\n")
            tracker_file.flush() # FORCE write to disk
        else:
            print(f"Skipping batch starting at index {i} due to repeated failures.")
        
        # Respect the rate limit safety net
        time.sleep(SLEEP_TIME)

print("\nFinished processing all batches!")

# ---------------------------------------------------------
# 4. Generate the Final DataFrame
# ---------------------------------------------------------
# Table containing the URIs of all posts for which a valid call was made
calls_made_df = pd.DataFrame(list(processed_root_uris), columns=['processed_root_uri'])
print(f"Total processed URIs logged: {len(calls_made_df)}")

Loaded 2292372 previously processed URIs from disk.
Extracting unique URIs using PySpark...


Found 0 new unique URIs to fetch.

Finished processing all batches!
Total processed URIs logged: 2292372


In [26]:
# ---------------------------------------------------------
# 1. Setup Persistent Storage Paths & Constants
# ---------------------------------------------------------
TRACKING_FILE = "/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/processed_root_uris.txt"
RESULTS_FILE = "/mnt/d/Bluesky-Reddit-BigDataProject/Bluesky_data/silver/getPosts/posts_results.jsonl" 
BSKY_API_URL = "https://public.api.bsky.app/xrpc/app.bsky.feed.getPosts"

BATCH_SIZE = 25
# 0.15 seconds yields ~2000 requests per 5 mins (safely under the 3000 limit)
SLEEP_TIME = 0.05

processed_root_uris = set()

# Load existing URIs if the script is restarting after a crash
if os.path.exists(TRACKING_FILE):
    with open(TRACKING_FILE, "r") as f:
        processed_root_uris = set(line.strip() for line in f if line.strip())
    print(f"Loaded {len(processed_root_uris)} previously processed URIs from disk.")

# ---------------------------------------------------------
# 2. Batch Fetching Helper Function (with 429 Retry Logic)
# ---------------------------------------------------------
def fetch_posts_batch(uris_batch, retries=3):
    """Fetches a batch of posts with built-in retry logic for rate limits."""
    for attempt in range(retries):
        try:
            params = {'uris': uris_batch}
            response = requests.get(BSKY_API_URL, params=params)
            
            # Handle rate limiting explicitly
            if response.status_code == 429:
                # Use the API's requested wait time, default to 5 seconds if missing
                wait = int(response.headers.get("Retry-After", 5))
                print(f"Rate limited (429). Sleeping {wait} seconds...")
                time.sleep(wait)
                continue # Try the same batch again

            # For all other HTTP errors
            response.raise_for_status()
            return response.json().get('posts', [])

        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2) # Brief pause before retrying a general network failure

    # If it fails after all retries
    return None

# ---------------------------------------------------------
# 3. Main Processing Logic
# ---------------------------------------------------------
print("Extracting unique URIs using PySpark...")

# Step A: Collect all unique URIs using Spark's distributed power
# This safely navigates the nested JSON/Struct schema, finds unique values, and drops nulls/errors
try:
    unique_uris_df = root_only_df.select("current_post_uri").distinct().dropna()
    
    # .collect() pulls the unique list from the cluster back to your main driver node
    all_root_uris = [row.current_post_uri for row in unique_uris_df.collect()]
    
except Exception as e:
    print(f"Error extracting URIs with Spark. Ensure schema matches: {e}")
    all_root_uris = []

# Filter out the URIs we've already processed in a previous crash/run
pending_uris = [uri for uri in all_root_uris if uri not in processed_root_uris]

print(f"Found {len(pending_uris)} new unique URIs to fetch.")

# ... Continue with Step B (the batch fetching loop) exactly as it was ...

# Step B: Process in chunks of 25 with safe disk-writing
with open(TRACKING_FILE, "a") as tracker_file, open(RESULTS_FILE, "a") as results_file:
    
    for i in range(0, len(pending_uris), BATCH_SIZE):
        batch = pending_uris[i:i + BATCH_SIZE]
        print(f"Fetching batch of {len(batch)} URIs ({i} to {i + len(batch)})...")
        
        posts_data = fetch_posts_batch(batch)
        
        # ONLY save and mark as processed if the fetch was completely successful
        if posts_data is not None:
            
            # 1. Save the actual JSON post data to disk immediately
            for post in posts_data:
                results_file.write(json.dumps(post) + "\n")
            results_file.flush() # FORCE write to disk
            
            # 2. Update tracking set and save URIs to disk immediately
            for uri in batch:
                processed_root_uris.add(uri)
                tracker_file.write(f"{uri}\n")
            tracker_file.flush() # FORCE write to disk
        else:
            print(f"Skipping batch starting at index {i} due to repeated failures.")
        
        # Respect the rate limit safety net
        time.sleep(SLEEP_TIME)

print("\nFinished processing all batches!")

# ---------------------------------------------------------
# 4. Generate the Final DataFrame
# ---------------------------------------------------------
# Table containing the URIs of all posts for which a valid call was made
calls_made_df = pd.DataFrame(list(processed_root_uris), columns=['processed_root_uri'])
print(f"Total processed URIs logged: {len(calls_made_df)}")

Loaded 2292372 previously processed URIs from disk.
Extracting unique URIs using PySpark...


----------------------------------------                                        
Exception occurred during processing of request from ('127.0.0.1', 51142)
Traceback (most recent call last):
  File "/usr/lib/python3.12/socketserver.py", line 318, in _handle_request_noblock
    self.process_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 349, in process_request
    self.finish_request(request, client_address)
  File "/usr/lib/python3.12/socketserver.py", line 362, in finish_request
    self.RequestHandlerClass(request, client_address, self)
  File "/usr/lib/python3.12/socketserver.py", line 761, in __init__
    self.handle()
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 303, in handle
    poll(accum_updates)
  File "/mnt/d/Bluesky-Reddit-BigDataProject/wsl_venv/lib/python3.12/site-packages/pyspark/accumulators.py", line 272, in poll
    if self.rfile in r and func():
                  

Found 1448142 new unique URIs to fetch.
Fetching batch of 25 URIs (0 to 25)...
Fetching batch of 25 URIs (25 to 50)...
Fetching batch of 25 URIs (50 to 75)...
Fetching batch of 25 URIs (75 to 100)...
Fetching batch of 25 URIs (100 to 125)...
Fetching batch of 25 URIs (125 to 150)...
Fetching batch of 25 URIs (150 to 175)...
Fetching batch of 25 URIs (175 to 200)...
Fetching batch of 25 URIs (200 to 225)...
Fetching batch of 25 URIs (225 to 250)...
Fetching batch of 25 URIs (250 to 275)...
Fetching batch of 25 URIs (275 to 300)...
Fetching batch of 25 URIs (300 to 325)...
Fetching batch of 25 URIs (325 to 350)...
Fetching batch of 25 URIs (350 to 375)...
Fetching batch of 25 URIs (375 to 400)...
Fetching batch of 25 URIs (400 to 425)...
Fetching batch of 25 URIs (425 to 450)...
Fetching batch of 25 URIs (450 to 475)...
Fetching batch of 25 URIs (475 to 500)...
Fetching batch of 25 URIs (500 to 525)...
Fetching batch of 25 URIs (525 to 550)...
Fetching batch of 25 URIs (550 to 575)...
Fe

KeyboardInterrupt: 